# 365 Probabilidades · Dia #067
## Qual a probabilidade de a hora da consulta mudar o seu tratamento?

**Tipo:** Comportamental
**Data de publicação:** 2026-08-19
**Ferramenta:** Python
**Decisão analisada:** A ordem em que eu decido muda o que eu decido?
**Hashtag:** #365Probabilidades #Dia067

---

### 📖 A História

Um dia de trabalho não é um bloco. É uma fila.

Você decide o que responder primeiro, quem espera, o que aprova, o que devolve,
o que deixa para amanhã. Nenhuma dessas decisões é grande sozinha. Somadas, elas
são o dia inteiro.

E a suposição que sustenta qualquer agenda é que a décima decisão sai igual à
primeira. Que a pessoa que decide às 16h é a mesma que decidiu às 9h.

Essa suposição é testável, e o lugar mais fácil de testar é onde a decisão fica
registrada com hora marcada e existe um padrão externo dizendo o que era certo
fazer. Consultório médico é exatamente isso.

Se a mesma queixa, com o mesmo perfil de paciente, receber conduta diferente às 8h
e às 11h, sobra pouca explicação simpática.

---

### 📚 O Conceito: um efeito que morreu no laboratório

Fadiga de decisão é prima de um conceito chamado esgotamento do ego, a ideia de que
o autocontrole é um recurso que se gasta.

E aqui o dia precisa começar admitindo o problema: **o esgotamento do ego não
sobreviveu à replicação.** Hagger e colegas, com 23 laboratórios e N=2.141, e Vohs e
colegas, com N=3.531, não encontraram o efeito.

Tem mais. O estudo mais citado do mundo sobre fadiga de decisão, o dos juízes que
negariam mais liberdade condicional conforme a sessão avança, foi mostrado como
provável artefato de agendamento: a ordem dos casos não era aleatória. Ele está fora
deste notebook, e deveria estar fora de qualquer texto sobre o assunto.

Sobrou o quê, então?

Sobrou o que se mede em campo, com desfecho administrativo objetivo, dentro do mesmo
profissional ao longo do próprio turno, e com o perfil das pessoas atendidas
controlado. É outro tipo de evidência, e é o único que este dia usa.

Uma distinção que precisa ficar clara: mostrar o **padrão** não é o mesmo que provar o
**mecanismo**. O padrão aqui é sólido. O mecanismo psicológico continua em disputa, e
"o médico está atrasado no cronograma" é uma explicação concorrente honesta.

---

### 🧮 O Modelo

Dois estudos de registro eletrônico de saúde, com desfechos objetivos e horário
marcado.

**Fontes:**
- Linder, J. A., Doctor, J. N., Friedberg, M. W., Reyes Nieva, H., Birks, C.,
  Meeker, D. & Fox, C. R., 2014 · *JAMA Internal Medicine* 174(12), 2029-2031 ·
  **21.867 consultas** por infecção respiratória aguda, **204 clínicos, 23 práticas**,
  adultos de 18 a 64 anos, maio de 2011 a setembro de 2012 · 44% resultaram em
  prescrição de antibiótico · razões de chances ajustadas em relação à primeira hora
  da sessão: **2ª hora 1,01 [0,91 · 1,13]**, **3ª hora 1,14 [1,02 · 1,27]**,
  **4ª hora 1,26 [1,13 · 1,41]**, p < 0,001 para tendência linear ·
  **controle:** a proporção de consultas em que o antibiótico era às vezes indicado
  não variou significativamente de hora em hora (p = 0,64)
- Hsiang, E. Y., Mehta, S. J., Small, D. S., Rareshide, C. A. L., Snider, C. K.,
  Day, S. C. & Patel, M. S., 2019 · *JAMA Network Open* 2(5), e193403 · 33 práticas
  na Pensilvânia e em Nova Jersey, 2014 a 2016 · **19.254 elegíveis** para rastreio de
  mama e **33.468** para colorretal · rastreio colorretal concluído por **28%** dos
  pacientes atendidos na hora das 8h contra **18%** dos atendidos às 17h ou depois
- Hagger, M. S. et al., 2016 · 23 laboratórios, **N=2.141** · e Vohs, K. D. et al.,
  2021 · **N=3.531** · falha de replicação do esgotamento do ego, usados aqui como
  contrapeso declarado

**Fora do modelo, deliberadamente:** Danziger et al., 2011, sobre juízes e liberdade
condicional. Provável artefato de agendamento.

**Nota metodológica sobre o fator ×0.80:** não se aplica. São registros eletrônicos de
saúde e faturamento, com desfecho objetivo de prescrição e de realização de exame.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print("Bibliotecas carregadas")

In [ ]:
# --- DADOS DA LITERATURA ---

# Linder, Doctor, Friedberg, Reyes Nieva, Birks, Meeker & Fox, 2014
# JAMA Internal Medicine 174(12), 2029-2031
n_consultas   = 21_867
n_clinicos    = 204
n_praticas    = 23
p_prescricao  = 0.44          # proporcao global de consultas com antibiotico

# Razoes de chances ajustadas em relacao a PRIMEIRA hora da sessao
horas         = [1, 2, 3, 4]
or_hora       = [1.00, 1.01, 1.14, 1.26]
ic_hora       = [None, (0.91, 1.13), (1.02, 1.27), (1.13, 1.41)]
p_tendencia   = 0.001         # p < 0,001 para tendencia linear

# Controle negativo do proprio estudo:
# a gravidade dos casos NAO muda ao longo da sessao
p_controle_indicacao = 0.64   # p do teste de variacao hora a hora

# Hsiang, Mehta, Small, Rareshide, Snider, Day & Patel, 2019
# JAMA Network Open 2(5), e193403
n_mama        = 19_254
n_colorretal  = 33_468
n_praticas_h  = 33
p_colo_8h     = 0.28          # rastreio colorretal concluido, consulta as 8h
p_colo_17h    = 0.18          # consulta as 17h ou depois

# Contrapeso declarado: o esgotamento do ego falhou em replicacao
n_hagger      = 2_141         # 23 laboratorios
n_vohs        = 3_531

aplica_fator_080 = False

print("=" * 70)
print("  DADOS - A HORA DA DECISAO")
print("=" * 70)
print(f"\n  Linder et al., 2014:")
print(f"  -> {n_consultas:,}".replace(",", ".") +
      f" consultas · {n_clinicos} clinicos · {n_praticas} praticas")
print(f"  -> Prescricao de antibiotico em {p_prescricao*100:.0f}% das consultas")
print(f"\n  Razao de chances ajustada, em relacao a 1a hora da sessao:")
for h, orr, ic in zip(horas, or_hora, ic_hora):
    if ic is None:
        print(f"  -> {h}a hora: {orr:.2f}  (referencia)")
    else:
        print(f"  -> {h}a hora: {orr:.2f}  IC 95% [{ic[0]:.2f} · {ic[1]:.2f}]")
print(f"  -> Tendencia linear: p < {p_tendencia}")
print(f"\n  CONTROLE NEGATIVO do proprio estudo:")
print(f"  -> A proporcao de casos em que o antibiotico era as vezes indicado")
print(f"     NAO variou de hora em hora (p = {p_controle_indicacao})")
print(f"  -> Os pacientes das 11h eram iguais aos das 8h.")
print(f"\n  Hsiang et al., 2019 ({n_praticas_h} praticas):")
print(f"  -> Elegiveis: {n_mama:,}".replace(",", ".") + " (mama) e " +
      f"{n_colorretal:,}".replace(",", ".") + " (colorretal)")
print(f"  -> Rastreio colorretal concluido: {p_colo_8h*100:.0f}% as 8h"
      f" contra {p_colo_17h*100:.0f}% as 17h ou depois")
print(f"\n  Contrapeso (esgotamento do ego, falha de replicacao):")
print(f"  -> Hagger et al., 2016: 23 laboratorios, N={n_hagger:,}".replace(",", "."))
print(f"  -> Vohs et al., 2021: N={n_vohs:,}".replace(",", "."))
print(f"\n  Fator x0.80 aplicado: {aplica_fator_080}")
print("=" * 70)

In [ ]:
# --- O MODELO ---
# Assinatura estatistica: razao de chances ajustada por hora, com IC 95%.
# Depois, a traducao da razao de chances em probabilidade absoluta.

Z = stats.norm.ppf(0.975)


def prob_de(odds):
    return odds / (1 + odds)


def odds_de(p):
    return p / (1 - p)


# A razao de chances so vira probabilidade se houver uma referencia.
# O artigo publica a proporcao GLOBAL (44%), nao a da primeira hora.
# Premissa declarada: consultas distribuidas igualmente entre as quatro horas.
# Com ela, resolve-se qual p da 1a hora reproduz os 44% globais.
def media_das_horas(p1):
    o1 = odds_de(p1)
    return np.mean([prob_de(orr * o1) for orr in or_hora])


p_primeira = optimize.brentq(lambda p: media_das_horas(p) - p_prescricao, 0.20, 0.70)
o1 = odds_de(p_primeira)
p_por_hora = np.array([prob_de(orr * o1) for orr in or_hora])
dif_pp = (p_por_hora[-1] - p_por_hora[0]) * 100

# Incerteza da 4a hora propagada a partir do IC publicado
ep_log_or4 = (np.log(ic_hora[3][1]) - np.log(ic_hora[3][0])) / (2 * Z)
p4_ic = [prob_de(np.exp(np.log(or_hora[3]) + s * Z * ep_log_or4) * o1) for s in (-1, 1)]

# Sensibilidade: e se a 1a hora nao for 41,7%?
grade_p1 = np.linspace(0.30, 0.55, 6)
sensibilidade = [(p1, prob_de(or_hora[3] * odds_de(p1)) - p1) for p1 in grade_p1]

# Traducao para escala humana
consultas_por_clinico = n_consultas / n_clinicos
extra_por_100 = dif_pp

print("=" * 70)
print("  MODELO - DA RAZAO DE CHANCES PARA A VIDA REAL")
print("=" * 70)
print(f"\n  1) O gradiente publicado (ajustado):")
print(f"  -> 2a hora: praticamente igual a 1a (IC cruza 1)")
print(f"  -> 3a hora: 1,14  IC 95% [1,02 · 1,27]  (IC nao cruza 1)")
print(f"  -> 4a hora: 1,26  IC 95% [1,13 · 1,41]  (IC nao cruza 1)")
print(f"  -> O efeito nao e imediato. Ele aparece na metade da sessao.")
print(f"\n  2) Traducao em probabilidade absoluta")
print(f"     PREMISSA DECLARADA: consultas igualmente distribuidas nas 4 horas,")
print(f"     calibradas para reproduzir os {p_prescricao*100:.0f}% globais publicados.")
for h, p in zip(horas, p_por_hora):
    print(f"  -> {h}a hora: {p*100:.1f}% de chance de sair com antibiotico")
print(f"  -> Diferenca da 1a para a 4a hora: {dif_pp:.1f} pontos percentuais")
print(f"  -> A 4a hora com o IC da razao de chances:"
      f" [{p4_ic[0]*100:.1f}% · {p4_ic[1]*100:.1f}%]")
print(f"\n  3) A conclusao NAO depende da premissa:")
for p1, d in sensibilidade:
    print(f"  -> Se a 1a hora fosse {p1*100:.0f}%, a 4a seria"
          f" {(p1+d)*100:.1f}%  (+{d*100:.1f} pp)")
print(f"\n  4) Em escala humana:")
print(f"  -> Cerca de {extra_por_100:.0f} prescricoes a mais a cada 100 consultas,")
print(f"     so por estar no fim da sessao em vez do comeco.")
print(f"  -> Media de {consultas_por_clinico:.0f} consultas de IRA por clinico no periodo.")
print(f"\n  5) O que sustenta a leitura:")
print(f"  -> Comparacao DENTRO do mesmo clinico, ao longo do proprio turno")
print(f"  -> Gravidade dos casos estavel hora a hora (p = {p_controle_indicacao})")
print(f"  -> Desfecho objetivo, registrado, nao autorrelatado")
print(f"  -> Mas: associacao, nao causa. E o mecanismo continua em disputa.")
print("=" * 70)

In [ ]:
# --- VISUALIZACAO ---

def br(n):
    return f"{n:,}".replace(",", ".")


DOURADO = '#c8a84b'
VERMELHO = '#c0392b'
VERDE = '#2a8a82'
CINZA = '#6b6a64'

# GRAFICO 1 - Assinatura: razao de chances por hora da sessao
fig1, ax1 = plt.subplots(figsize=(12, 8))

y = np.arange(len(horas))[::-1]
for yi, h, orr, ic in zip(y, horas, or_hora, ic_hora):
    cor = CINZA if ic is None else (DOURADO if ic[0] < 1 else VERMELHO)
    if ic is not None:
        ax1.plot([ic[0], ic[1]], [yi, yi], color=cor, linewidth=3.2,
                 solid_capstyle='round')
        for lim in ic:
            ax1.plot([lim, lim], [yi - 0.09, yi + 0.09], color=cor, linewidth=3.2)
    ax1.scatter([orr], [yi], s=230, color=cor, zorder=3)
    rotulo = f'{orr:.2f}'.replace('.', ',') + ('  (referência)' if ic is None else '')
    ax1.text(orr, yi + 0.24, rotulo, ha='center', fontsize=15,
             fontweight='bold', color=cor)

ax1.axvline(x=1.0, color='#333', linestyle='--', linewidth=1.6)
ax1.set_yticks(y)
ax1.set_yticklabels([f'{h}ª hora da sessão' for h in horas], fontsize=13)
ax1.set_xlim(0.82, 1.52)
ax1.set_ylim(-0.6, y.max() + 0.7)
ax1.set_xlabel('Razão de chances ajustada de prescrever antibiótico')
ax1.set_title('Quanto mais tarde na sessão, maior a chance de sair com antibiótico\n'
              f'Linder et al., 2014 · {br(n_consultas)} consultas · '
              f'{n_clinicos} clínicos',
              fontsize=14, pad=18)
ax1.text(0.5, -0.12, 'Tendência linear ao longo da sessão: p < 0,001',
         transform=ax1.transAxes, ha='center', fontsize=12, color=DOURADO,
         fontweight='bold')

plt.figtext(0.5, 0.005,
            'Fonte: Linder et al., 2014, JAMA Internal Medicine  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-067-grafico-01-hora-a-hora.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 1 salvo")

# GRAFICO 2 - Traducao em probabilidade e o controle negativo
fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(14, 7),
                                  gridspec_kw={'width_ratios': [1.5, 1]})

ax2a.plot(horas, p_por_hora * 100, marker='o', markersize=13, linewidth=3,
          color=VERMELHO)
for h, p in zip(horas, p_por_hora):
    ax2a.text(h, p * 100 + 0.5, f'{p*100:.1f}%'.replace('.', ','),
              ha='center', fontsize=14, fontweight='bold', color=VERMELHO)
ax2a.set_xticks(horas)
ax2a.set_xticklabels([f'{h}ª hora' for h in horas], fontsize=12)
ax2a.set_ylim(p_por_hora.min() * 100 - 3, p_por_hora.max() * 100 + 3)
ax2a.set_ylabel('Chance de sair da consulta com antibiótico (%)')
ax2a.set_title(f'Traduzindo em probabilidade\n'
               f'{dif_pp:.1f} pontos percentuais entre a 1ª e a 4ª hora'
               .replace('.', ','), fontsize=14, pad=14)
ax2a.text(0.5, 0.06,
          'Premissa declarada: consultas igualmente distribuídas nas quatro horas,\n'
          'calibradas para reproduzir os 44% globais publicados.',
          transform=ax2a.transAxes, ha='center', fontsize=10, color=CINZA,
          style='italic')

ax2b.bar([f'{h}ª' for h in horas], [1, 1, 1, 1], color=VERDE, alpha=0.55, width=0.6)
ax2b.set_ylim(0, 2.3)
ax2b.set_yticks([])
ax2b.set_title('O controle negativo\n'
               'A gravidade dos casos não muda ao longo da sessão',
               fontsize=14, pad=14)
ax2b.text(1.5, 1.95, f'p = 0,64', ha='center', fontsize=20,
          fontweight='bold', color=VERDE)
ax2b.text(1.5, 1.68, 'proporção de casos em que o antibiótico\n'
                     'era às vezes indicado, hora a hora',
          ha='center', fontsize=10, color=CINZA, style='italic')
ax2b.text(1.5, 1.22, 'os pacientes das 11h\neram iguais aos das 8h',
          ha='center', va='center', fontsize=13, color='#333', fontweight='bold')
ax2b.grid(False)

plt.figtext(0.5, 0.005,
            'Fonte: Linder et al., 2014, JAMA Internal Medicine  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-067-grafico-02-traducao.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 2 salvo")

# GRAFICO 3 - O mesmo padrao em outro desfecho
fig3, ax3 = plt.subplots(figsize=(12, 8))

rotulos3 = ['Consulta na hora\ndas 8h', 'Consulta às 17h\nou depois']
valores3 = [p_colo_8h * 100, p_colo_17h * 100]
barras3 = ax3.bar(rotulos3, valores3, color=[VERDE, VERMELHO], alpha=0.9, width=0.45)

for barra, valor in zip(barras3, valores3):
    ax3.text(barra.get_x() + barra.get_width() / 2, valor + 0.7,
             f'{valor:.0f}%', ha='center', fontsize=24, fontweight='bold')

ax3.annotate('', xy=(1, p_colo_17h * 100), xytext=(0, p_colo_8h * 100),
             arrowprops=dict(arrowstyle='->', color=DOURADO, lw=2.5))
ax3.text(0.5, (p_colo_8h + p_colo_17h) / 2 * 100 + 1.5,
         f'{(p_colo_8h - p_colo_17h)*100:.0f} pontos percentuais',
         ha='center', fontsize=14, fontweight='bold', color=DOURADO)

ax3.set_ylim(0, 34)
ax3.set_ylabel('Rastreio de câncer colorretal concluído em até 1 ano (%)')
ax3.set_title('O mesmo padrão, outro desfecho, outra rede de clínicas\n'
              f'Hsiang et al., 2019 · {n_praticas_h} práticas · '
              f'{br(n_colorretal)} pacientes elegíveis',
              fontsize=14, pad=18)
ax3.text(0.5, -0.13,
         'Apenas os dois horários publicados estão desenhados. '
         'Nenhum ponto intermediário foi estimado.',
         transform=ax3.transAxes, ha='center', fontsize=11, color=CINZA,
         style='italic')

plt.figtext(0.5, 0.005,
            'Fonte: Hsiang et al., 2019, JAMA Network Open 2(5)  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-067-grafico-03-outro-desfecho.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 3 salvo")

### 💡 O Insight

Vinte e um mil oitocentas e sessenta e sete consultas. Duzentos e quatro médicos.
A mesma queixa, o mesmo tipo de paciente, o mesmo profissional.

**Na quarta hora da sessão, a chance de sair de lá com uma receita de antibiótico é
26% maior do que na primeira** (razão de chances 1,26, IC 95% [1,13 · 1,41]).

Em probabilidade absoluta isso são cerca de **6 pontos percentuais**, ou seja, algo
como seis receitas a mais a cada cem consultas, só por causa do horário.

E o número que fecha a porta para a explicação simpática: a proporção de casos em que
o antibiótico era de fato às vezes indicado **não mudou ao longo da sessão**, com
p = 0,64. Os pacientes das 11h eram iguais aos das 8h. O que mudou foi quem decidia.

Numa rede diferente, com outro desfecho, o padrão reaparece. Rastreio de câncer
colorretal concluído por 28% de quem foi atendido às 8h, contra 18% de quem foi
atendido às 17h ou depois.

Agora a parte que me obriga a segurar o entusiasmo.

O conceito psicológico por trás disso, o esgotamento do ego, **não sobreviveu à
replicação** em laboratório. E o estudo mais famoso sobre fadiga de decisão, o dos
juízes, é provável artefato de agendamento e não entrou aqui.

O que sobrevive é mais modesto e mais útil: **o padrão temporal existe, é grande, e
aparece em registros objetivos.** Por que ele existe é outra pergunta. Pode ser
desgaste cognitivo, pode ser o médico correndo atrás do atraso, pode ser as duas
coisas.

O que dá para dizer sem exagero é isto: a hora em que uma decisão acontece carrega
informação sobre o resultado dela. Não porque a pessoa piora, mas porque nenhuma
agenda é feita de decisões independentes.

E, olhando de fora, dá para usar isso. Se alguma decisão sua importa de verdade, ela
não deveria ser a décima quarta do dia.

*Que decisão importante você deixa, sistematicamente, para o fim do dia?*

---

### ⚠️ Limitações do Modelo

- **Associação, não causa.** Ninguém sorteou o horário das consultas. O estudo é
  observacional, ainda que ajustado e comparando dentro do mesmo clínico.
- **O mecanismo é disputado.** Fadiga de decisão é uma hipótese entre outras. Os
  próprios autores do estudo de rastreio levantam o atraso no cronograma como
  explicação concorrente, e o esgotamento do ego falhou em replicações grandes
  (Hagger et al., 2016, 23 laboratórios, N=2.141; Vohs et al., 2021, N=3.531).
- **A tradução de razão de chances em probabilidade exige uma premissa.** O artigo
  publica a proporção global de 44%, não a da primeira hora. Assumi distribuição igual
  de consultas entre as quatro horas para calibrar. A conclusão qualitativa não muda
  em nenhum ponto da faixa testada, e a análise de sensibilidade está no notebook.
- A segunda hora não difere da primeira: o intervalo cruza 1. O efeito aparece do meio
  da sessão em diante, e descrevê-lo como crescimento contínuo desde a primeira
  consulta seria exagero.
- Do estudo de rastreio, apenas os dois horários publicados foram desenhados. Nenhum
  ponto intermediário foi estimado.
- Pacientes adultos de 18 a 64 anos, em redes específicas dos Estados Unidos, com
  organização de agenda em sessões de quatro horas. Generalização para outros sistemas
  de saúde não está testada.
- **Danziger et al., 2011, ficou fora de propósito.** É o estudo mais citado do tema e
  é provável artefato de agendamento.
- Fator ×0.80 não aplicado: registros eletrônicos de saúde e faturamento, com desfecho
  objetivo.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
